# Twitter User ID Lookup via SocialVec

This notebook:
1. Loads and concatenates the two CSV files from `data_preperation/data/`
2. Uses SocialVec to look up each account's Twitter user ID by screen name
3. Saves the enriched DataFrame with the `twitter_user_id` column

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import random
import ast
from collections.abc import Iterable

## Load and Concatenate CSV Files

In [3]:
data_dir = Path("data")

# read and concatenate any csv files in the data directory
csv_files = list(data_dir.glob("*.csv"))
df_list = [pd.read_csv(file) for file in csv_files]
df_categories = pd.concat(df_list, ignore_index=True)
print(f"Combined shape: {df_categories.shape}")

Combined shape: (503, 4)


## Initialize SocialVec

In [4]:
from socialvec.socialvec import SocialVec

sv = SocialVec()

✅  Initializing Model
✅  Load Metadata


## Look Up Twitter User IDs

Strip the `@` prefix from each screen name and query SocialVec.
Users not found in SocialVec will be filled with `NaN`.

In [5]:
def get_user_id(screen_name: str) -> float:
    """Return the Twitter user ID for a screen name, or NaN if not found."""
    username = screen_name.lstrip("@")
    try:
        return sv.get_userid(username)
    except Exception:
        return np.nan


df_categories["twitter_user_id"] = df_categories["Twitter Account"].apply(get_user_id)

found = df_categories["twitter_user_id"].notna().sum()
total = len(df_categories)
print(f"Found user IDs for {found}/{total} accounts ({found/total:.1%})")

Found user IDs for 501/503 accounts (99.6%)


## Inspect Results

In [12]:
def get_embeddings(user_id: str) -> np.ndarray:
    """Return the embedding vector for a Twitter user ID, or None if not found."""
    try:
        return sv[user_id]
    except Exception:
        return None

df_categories['sv'] = df_categories['twitter_user_id'].apply(get_embeddings)

In [ ]:
# Accounts not found in SocialVec
missing = df_categories[df_categories["twitter_user_id"].isna()][["Category", "Twitter Account", "Full Name"]]
print(f"Missing ({len(missing)} accounts):")
missing

Missing (2 accounts):


,Category,Twitter Account,Full Name
482,🗳️ Politics,@RonDeSantis,Ron DeSantis
500,"🌱 Gardening, Agriculture & Homesteading",@BurpeeCo,Burpee Gardening


## Save Enriched CSV

In [ ]:
df_categories = df_categories.dropna(subset=["twitter_user_id"])

In [16]:
df_categories = df_categories[~df_categories['sv'].isna()]

In [18]:
output_path = data_dir / "curated_twitter_accounts_with_categories_and_ids.xlsx"
df_categories.to_excel(output_path, index=False)
print(f"Saved to {output_path}")

Saved to data/curated_twitter_accounts_with_categories_and_ids.xlsx


# Select users who follow accounts from the new account listm

In [ ]:
df_all = pd.read_pickle('/Volumes/itaa_mla_data/mla/users/nlotan/mla/sv/data/all_usa_users.pkl', compression='gzip')
df_all = df_all[['user_id','screen_name','friends_ids']]

In [ ]:
def flatten(xs):
    """Recursively flatten nested iterables (except strings)."""
    for x in xs:
        if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

def parse_friends_ids(val):
    if pd.isna(val):
        return []

    try:
        parsed = ast.literal_eval(val)
    except Exception:
        return []

    # Flatten and keep only numeric IDs
    return [int(x) for x in flatten(parsed) if str(x).isdigit()]

# Parse to 1‑D list
df_all["friends_ids_list"] = df_all["friends_ids"].progress_apply(parse_friends_ids)



In [ ]:
# Explode to one row per (user_id, friend_id)
df_exploded = (
    df_all
    .explode("friends_ids_list")
    .rename(columns={"friends_ids_list": "friend_id"})
    .loc[:, ["user_id", 'screen_name', "friend_id"]]
    .dropna(subset=["friend_id"])
)

In [ ]:
potential_users = df_exploded[df_exploded['friend_id'].isin(df_categories['twitter_user_id'].values)]['screen_name'].unique()
random_sample = random.sample(list(potential_users), 20000)
df_sample = df_all[df_all['screen_name'].isin(random_sample)]
df_sample=df_sample[~df_sample['description'].isna()]

df_sample["friends_ids_list"] = df_sample["friends_ids"].progress_apply(parse_friends_ids)
df_sample["friends_ids_len"] = df_sample["friends_ids_list"].apply(lambda x: len(x))

df_sample["friends_ids_len"].describe()

In [ ]:
# we choose the range based on the describe statistics, where we will have enough meaningful data, but also filter out outliers with too many or too few friends as they may be bots (filtering below 75th percentile and above 99th percentile)
df_sample[(df_sample["friends_ids_len"]>40)&(df_sample["friends_ids_len"]<1500)]

In [ ]:
# create a new column with the length of the intersection for the friends_ids_list and the df_categories['twitter_user_id'].values
df_sample["friends_ids_intersection_len"] = df_sample["friends_ids_list"].apply(lambda x: len(set(x) & set(df_categories['twitter_user_id'].values)))

In [ ]:
# now keep in df_sample only the users with at least 3 friends in the df_categories, which is the 25th precentile of the friends_ids_intersection_len distribution
df_sample = df_sample[df_sample["friends_ids_intersection_len"] >= 3]

# we remain with 10419 users.

In [20]:
df = pd.read_pickle('/Users/nlotan/code/university/personal_llm_vc/data/persona_details_v3.pkl')

In [21]:
df

,user_id,screen_name,description,follows_list,sv
0,84202663,JessicaMBailey_,# **Profile: Jessica Martin-Bailey**\n\n---\n\...,"[nickkroll, MTV, vmas, RecordingAcad, caseykfr...","[-0.090021245, -0.14723428, -0.048477273, 0.00..."
1,806956645,CoolNameGuy91,# Aiden Nakamura - The Pop Culture Enthusiast ...,"[erikaishii, BrennanLM, MicaBurton, SeanRossSa...","[-0.10349321, 0.15659052, -0.009157364, 0.2650..."
2,806956645,CoolNameGuy91,# **Profile Dossier: Alex Donovan**\n\n## **Pe...,"[erikaishii, BrennanLM, MicaBurton, SeanRossSa...","[-0.10349321, 0.15659052, -0.009157364, 0.2650..."
3,620429236,ItsMarkMorgan,# **Alex Montgomery: The Energetic Red Sox Ent...,"[Mythical, lilsasquatch66, MyFavMurder, hen_ea...","[0.012351625, -0.0075871036, 0.035785355, 0.20..."
4,3254803112,luckynik30,# Richard (Rick) Martinez \n**Age:** 45 \n**O...,"[TomCottonAR, elonmusk, GeorgiaLogCabin, aReal...","[0.32099348, -0.22670253, -0.12614475, 0.26361..."
...,...,...,...,...,...
5793,95805581,SandiR_68,# Profile of Sandra Richards\n\n## Personal In...,"[Bodegacats_, MacFarlaneNews, DGlaucomflecken,...","[0.048960604, 0.05026407, -0.029063094, 0.1466..."
5794,630832318,Lasso_Truth,"# Digital Dossier: Alexander ""Alex"" Whitmore\n...","[LilNasX, Phil_Mattingly, RightWingWatch, bren...","[-0.18107887, 0.034921836, -0.09580742, -0.035..."
5795,1439452685615476738,CoffeiCorpse,"# Twitter User Profile: Alexandria ""Alex"" Hast...","[alanalda, Braves, Talkmaster, GAFollowers, EW...","[0.14648268, 0.034068737, -0.084774464, 0.1574..."
5796,347561125,keifer66,# Profile Dossier: Keegan Douglas\n\n## Biogra...,"[nedryun, CashApp, Highway_30, realTanyaTay, H...","[0.06976476, 0.0911537, 0.07019684, 0.27537775..."
